# Reasoning SLM - Final: Test Set Evaluation

## Introduction

The final numbers: every model on the full 1,319-problem GSM8K test set, untouched until now. The 499-problem validation subset was used to iterate and to choose the GRPO checkpoint, so its scores are slightly optimistic, the test set gives unbiased ones (±1.4 pts).

| Model | Prompt |
| --- | --- |
| `Qwen2.5-0.5B` base | 8-shot |
| + SFT | zero-shot |
| + SFT + GRPO (step 400) | zero-shot |
| `Qwen2.5-0.5B-Instruct` | chat, zero-shot |

The same harness and settings as every stage: greedy decoding, at most 512 new tokens, the same stop rules and batch size. All models run back to back in one session on one GPU, because small numerical differences (a different merge path or GPU) can flip a few greedy answers.

## Setup

Clone the repository so the shared code in `src/` can be imported, and install the requirements. No vLLM is needed. The SFT and GRPO models are loaded from Google Drive (`/content/drive/MyDrive/slm-post-training/models`). Results are written to `results/test/`. Download them after the run, because `/content` is wiped when the runtime disconnects.

In [1]:
import os

REPO_URL = "https://github.com/JuaniLlaberia/slm-post-training.git"
REPO_DIR = "/content/slm-post-training"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

/content/slm-post-training


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Imports

In [3]:
from src.config import BASE_MODEL, INSTRUCT_MODEL, RESULTS_DIR
from src.data.gsm8k import load_gsm8k
from src.evaluation.harness import evaluate, save_results
from src.prompts.few_shot import EXAMPLES, STOP, build_instruct_prompt, build_prompt

DRIVE_DIR = "/content/drive/MyDrive/slm-post-training"
SFT_MODEL = f"{DRIVE_DIR}/models/sft-merged"
GRPO_MODEL = f"{DRIVE_DIR}/models/grpo_full-step400-merged"
TEST_DIR = RESULTS_DIR / "test"

## Data

Only the test split is used: 1,319 problems, never seen in training or used for any choice.

In [4]:
_, _, test_dataset = load_gsm8k()

questions = test_dataset["question"]
golds = test_dataset["gold"]

print(f"Test dataset size: {len(test_dataset)}")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Test dataset size: 1319


## Evaluation

### Base Model, 8-shot

In [5]:
base_rows = evaluate(BASE_MODEL, lambda q, _: build_prompt(q, examples=EXAMPLES), questions, golds, stop=STOP)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen/Qwen2.5-0.5B: acc=36.0%  no-answer=6.1%  hit-limit=4.1%


In [6]:
save_results(base_rows, TEST_DIR / "base_8shot_test.json")

### SFT

In [7]:
sft_rows = evaluate(SFT_MODEL, lambda q, _: build_prompt(q), questions, golds, stop=STOP)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/content/drive/MyDrive/slm-post-training/models/sft-merged: acc=35.0%  no-answer=3.0%  hit-limit=3.0%


In [8]:
save_results(sft_rows, TEST_DIR / "sft_0shot_test.json")

### SFT + GRPO

In [9]:
grpo_rows = evaluate(GRPO_MODEL, lambda q, _: build_prompt(q), questions, golds, stop=STOP)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/content/drive/MyDrive/slm-post-training/models/grpo_full-step400-merged: acc=47.7%  no-answer=0.6%  hit-limit=0.6%


In [10]:
save_results(grpo_rows, TEST_DIR / "grpo_full_step400_0shot_test.json")

### Instruct Model

As in Stage 1, the instruct model also gets a lenient score: the last number of each no-answer output that stopped on its own, which catches its plain-text final answers ("Final Answer: 3 months").

In [11]:
instruct_rows = evaluate(INSTRUCT_MODEL, build_instruct_prompt, questions, golds, stop=STOP)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-0.5B-Instruct: acc=38.8%  no-answer=24.0%  hit-limit=5.5%


In [12]:
save_results(instruct_rows, TEST_DIR / "instruct_0shot_test.json")

In [13]:
import re

def last_number(text):
    nums = re.findall(r"-?[\d,]*\.?\d+", text)
    return float(nums[-1].replace(",", "")) if nums else None

instruct_lenient = [r["correct"] or (r["extracted"] is None and not r["hit_limit"] and last_number(r["output"]) == r["gold"])
                    for r in instruct_rows]
print(f"instruct, lenient: {sum(instruct_lenient) / len(instruct_lenient):.1%}")

instruct, lenient: 46.9%


## Comparison

Accuracy, no-answer and hit-limit rates for every model, then GRPO against each one problem by problem, on the same 1,319 problems: `only_grpo` and `only_other` count the problems only one of them solves. `z` = (only_grpo − only_other) / √(only_grpo + only_other) is McNemar's paired test: if the two models were equally good, each disagreement would be a coin flip, and |z| > 2 means a gap this large would happen by chance less than 5% of the time.

In [14]:
import pandas as pd

models = {"base 8-shot": base_rows, "sft": sft_rows, "sft + grpo": grpo_rows, "instruct": instruct_rows}

print(pd.DataFrame([{"model": name,
                     "acc": sum(r["correct"] for r in rows) / len(rows),
                     "no_answer": sum(r["extracted"] is None for r in rows) / len(rows),
                     "hit_limit": sum(r["hit_limit"] for r in rows) / len(rows)}
                    for name, rows in models.items()]).to_string(index=False))

grpo_correct = [r["correct"] for r in grpo_rows]
others = {"base 8-shot": [r["correct"] for r in base_rows],
          "sft": [r["correct"] for r in sft_rows],
          "instruct": [r["correct"] for r in instruct_rows],
          "instruct (lenient)": instruct_lenient}

paired = []
for name, correct in others.items():
    only_grpo = sum(g and not o for g, o in zip(grpo_correct, correct))
    only_other = sum(o and not g for g, o in zip(grpo_correct, correct))
    paired.append({"vs": name,
                   "only_grpo": only_grpo,
                   "only_other": only_other,
                   "z": (only_grpo - only_other) / max(1, only_grpo + only_other) ** 0.5})

print(pd.DataFrame(paired).to_string(index=False))

      model      acc  no_answer  hit_limit
base 8-shot 0.360121   0.061410   0.040940
        sft 0.349507   0.030326   0.030326
 sft + grpo 0.476876   0.006065   0.006065
   instruct 0.388173   0.240334   0.055345
                vs  only_grpo  only_other        z
       base 8-shot        255         101 8.161984
               sft        258          90 9.005745
          instruct        269         152 5.702231
instruct (lenient)        186         175 0.578947


| Model | Prompt | Test (1,319) | No-answer | Validation (499) |
| --- | --- | --- | --- | --- |
| Base | 8-shot | 36.0% | 6.1% | 46.9% |
| Base + SFT | zero-shot | 35.0% | 3.0% | 46.5% |
| Base + SFT + GRPO (step 400) | zero-shot | 47.7% | 0.6% | 64.7% |
| Instruct | chat, zero-shot | 38.8% (46.9% lenient) | 24.0% | 53.3% (61.5% lenient) |

- **SFT changed format only:** 35.0% against base 8-shot's 36.0%.
- **GRPO added real math ability:** +12.7 points over SFT (`z` = 9.0; 258 problems only GRPO solves, 90 only SFT). About 70% of the validation gain (12.7 of 18.2 points) carries over to unseen problems.
- **GRPO against instruct:** clearly above it as scored (`z` = 5.7), and level with it once its format is fixed (47.7% against 46.9%, `z` = 0.6). GRPO closed the gap Stage 2 found between this model and Qwen's instruct model, but doesn't pass it. On validation, 64.7% against 61.5% looked like a lead; on test it isn't one.
- **Qwen's post-training improved the math:** instruct's lenient 46.9% is far above base's 36.0%, while as scored (38.8%) most of the gain hides behind its unformatted answers, as on validation.

## Possible Contamination

Every model scores 11-17 points lower on test than on validation:

| Model | Validation | Test | Drop |
| --- | --- | --- | --- |
| Base, 8-shot | 46.9% | 36.0% | −10.9 |
| Base + SFT | 46.5% | 35.0% | −11.5 |
| Base + SFT + GRPO | 64.7% | 47.7% | −17.0 |
| Instruct | 53.3% | 38.8% | −14.5 |

The validation subset is a random part of GSM8K train, and train and test come from the same source, so a gap this size is far beyond noise (±2.2 and ±1.4). The most likely cause is that GSM8K train problems were in Qwen2.5's training data: validation problems were familiar, test problems are not. It fits the base model's outputs carrying instruction-tuning data from pretraining (Stage 1), and base and SFT hitting the token limit more often on test (4.1% and 3.0%, against 2.4% and 1.2% on validation). It can't be proven from accuracy alone.

Comparisons between models on the same problems still hold, and every validation conclusion survives on test except GRPO leading instruct's lenient score. GRPO drops the most: trained on the same train split, part of its validation gain likely came from making the model answer familiar problems more reliably, which doesn't carry over to unseen ones.